# DACIL-WESENSE Quantitative Analysis Pipeline

**Pipeline overview:**

1. Environment setup and batch folder discovery
2. Data loading: CSV telemetry and BDF ECG files
3. Data synchronisation and cleaning
4. Exploratory Data Analysis (EDA)
5. Unsupervised Machine Learning (PCA, K-Means, DBSCAN)
6. Per-patient output export
7. Master summary CSV and notebook export


## 1. Environment Setup


In [1]:
if True:
    !pip install -r requirements.txt

In [2]:
import logging
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import functions as fn
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Configure logging: messages appear both in the notebook and in a log file.
fn.setup_logging(log_file="pipeline.log", level=logging.INFO)
logger = logging.getLogger("main")

logger.info("Pipeline started.")


2026-03-04T19:01:51 | INFO     | main | Pipeline started.


## 2. Configuration

Set `DATA_ROOT` to the directory that contains one sub-folder per patient/trial.
All outputs are written under `OUTPUT_ROOT`.


In [3]:
# ----------------------------------------------------------------
# CONFIGURE THESE PATHS BEFORE RUNNING
# ----------------------------------------------------------------
DATA_ROOT: str = "data"          # root directory with patient sub-folders
OUTPUT_ROOT: str = "output"      # base output directory
# ----------------------------------------------------------------

output_base = Path(OUTPUT_ROOT)
output_base.mkdir(parents=True, exist_ok=True)

logger.info("Data root: %s", DATA_ROOT)
logger.info("Output root: %s", OUTPUT_ROOT)


2026-03-04T19:01:51 | INFO     | main | Data root: data
2026-03-04T19:01:51 | INFO     | main | Output root: output


## 3. Batch Folder Discovery

All immediate sub-directories of `DATA_ROOT` are treated as independent
patient/trial folders.  A `try-except` block around each folder ensures
that a single problematic folder does not abort the entire batch.


In [4]:
patient_folders = fn.discover_patient_folders(DATA_ROOT)
print(f"Found {len(patient_folders)} patient folder(s).")
for pf in patient_folders:
    print(" -", pf.name)

2026-03-04T19:01:51 | INFO     | functions | Discovered 1 patient folder(s) under 'data'.


Found 1 patient folder(s).
 - Patient 1


## 4. Batch Processing Loop

For each patient folder the pipeline:

1. Loads CSV telemetry and patient metadata.
2. Loads BDF ECG files (L1 and L2) and extracts summary ECG features.
3. Synchronises ECG features with the telemetry time series.
4. Saves the cleaned telemetry as a CSV.
5. Appends a summary row to the master list.

Any folder that raises an exception is logged and skipped gracefully.


In [ ]:
# Set to True to load cached ECG features from disk (skip BDF processing when available).
# Set to False to always reprocess from the BDF files.
USE_ECG_CHECKPOINT = True

master_records = []
processed_telemetry: dict = {}  # patient_id -> cleaned telemetry DataFrame

n_total = len(patient_folders)
for i, folder in enumerate(tqdm(patient_folders, desc="Processing patients"), start=1):
    patient_id = folder.name
    print(f"\n[{i}/{n_total}] ▶ Patient: {patient_id}")
    logger.info("Processing patient: %s", patient_id)

    try:
        # ── 4a. Load CSV ──────────────────────────────────────────────────
        csv_path = fn.find_csv_file(folder)
        if csv_path is None:
            raise FileNotFoundError(f"No CSV file found in {folder}")

        info_df, telemetry_df = fn.load_telemetry(csv_path)
        print(f"  Telemetry loaded: {telemetry_df.shape[0]} rows, {telemetry_df.shape[1]} columns")
        logger.info("  Telemetry shape: %s", telemetry_df.shape)

        # ── 4b. Load BDF ECG files ─────────────────────────────────────────
        l1_path, l2_path = fn.find_bdf_files(folder)
        ecg_features_list = []
        patient_out = output_base / patient_id

        for bdf_label, bdf_path in [("L1", l1_path), ("L2", l2_path)]:
            if bdf_path is None:
                print(f"  ECG {bdf_label}: not found")
                logger.warning("  %s BDF not found for %s.", bdf_label, patient_id)
                continue
            print(f"  ECG {bdf_label}: found ({bdf_path.name})")

            feats = None
            if USE_ECG_CHECKPOINT:
                feats = fn.load_ecg_checkpoint(patient_out, bdf_label)
                if feats is not None:
                    print(f"  ECG {bdf_label}: loaded from checkpoint")

            if feats is None:
                raw = fn.load_ecg(bdf_path)
                if raw is not None:
                    feats = fn.extract_ecg_features(raw)
                    fn.save_ecg_checkpoint(feats, patient_out, bdf_label)
                    print(f"  ECG {bdf_label}: processed and checkpoint saved")

            if feats is not None:
                feats = feats.copy()
                feats["bdf_label"] = bdf_label
                ecg_features_list.append(feats)

        ecg_features = (
            pd.concat(ecg_features_list, ignore_index=True)
            if ecg_features_list
            else pd.DataFrame()
        )

        # ── 4c. Synchronise ECG + telemetry ───────────────────────────────
        telemetry_df = fn.sync_ecg_with_telemetry(ecg_features, telemetry_df)

        # ── 4d. Save cleaned CSV ───────────────────────────────────────────
        fn.save_telemetry_csv(telemetry_df, patient_id, patient_out)

        # ── 4e. Build summary row ──────────────────────────────────────────
        summary_row = fn.build_patient_summary(
            patient_id, info_df, telemetry_df, ecg_features if not ecg_features.empty else None
        )
        master_records.append(summary_row)
        processed_telemetry[patient_id] = telemetry_df

        print(f"  Done")
        logger.info("  Successfully processed patient: %s", patient_id)

    except Exception as exc:
        print(f"  FAILED: {exc}")
        logger.error("  FAILED for patient '%s': %s", patient_id, exc, exc_info=True)

print(f"\nBatch complete. Successfully processed {len(master_records)} patient(s).")


## 5. Master Summary DataFrame

Compile all per-patient summary rows into a single master DataFrame and
save it as `output/master_summary.csv`.


In [ ]:
if master_records:
    summary_df = pd.DataFrame(master_records)
    master_csv = output_base / "master_summary.csv"
    summary_df.to_csv(master_csv, index=False)
    logger.info("Saved master summary: %s", master_csv)
    display(summary_df.head())
else:
    summary_df = pd.DataFrame()
    print("No patient data available. Check that DATA_ROOT contains valid folders.")


## 6. Exploratory Data Analysis

### 6a. Per-patient stage profiles

For each successfully processed patient we plot how the key physiological
metrics (VO2, HR, Power, VCO2) evolve across the five trial stages.
Figures are saved under `output/<patient_id>/`.


In [ ]:
EDA_METRICS = ["HR", "VO2", "VCO2", "Power", "RER", "SpO2"]

for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="EDA plots"):
    patient_out = output_base / patient_id
    fig = fn.plot_metrics_by_stage(
        telemetry_df,
        metrics=EDA_METRICS,
        patient_id=patient_id,
        output_dir=patient_out,
    )
    plt.show()
    plt.close(fig)


### 6b. Stage-level summary statistics

Mean and standard deviation of key metrics for each trial stage.


In [ ]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="Stage summaries"):
    print(f"\n--- Patient: {patient_id} ---")
    stage_summary = fn.compute_stage_summary(telemetry_df, metrics=EDA_METRICS)
    display(stage_summary)


### 6c. Batch aggregate visualisations

Aggregated views across all patients:

- Distribution of peak VO2 grouped by gender.
- BMI vs peak VO2 scatter plot.


In [ ]:
if not summary_df.empty:
    batch_figs = fn.plot_batch_aggregates(summary_df, output_dir=output_base)
    for fig in batch_figs:
        plt.show()
        plt.close(fig)
else:
    print("No summary data available for batch EDA.")


### 6d. Correlation heatmap

Pearson correlations between the key CPET metrics pooled across all patients.
Strong positive correlations (e.g. VO2–VCO2) confirm physiological coupling;
the RER column reveals how metabolic shift relates to ventilatory demand.


In [ ]:
CORR_CANDIDATES = [
    "HR", "VO2", "VCO2", "RER", "VE", "BF", "VTex",
    "SpO2", "VO2/HR", "PETCO2", "PETO2", "EqO2", "EqCO2",
]

if processed_telemetry:
    all_tele_corr = pd.concat(processed_telemetry.values(), ignore_index=True)
    present = [fn._find_column(all_tele_corr, [c]) for c in CORR_CANDIDATES]
    present = [c for c in present if c is not None]
    corr_df = all_tele_corr[present].apply(pd.to_numeric, errors="coerce").dropna(how="all")

    fig, ax = plt.subplots(figsize=(10, 8))
    import seaborn as sns
    sns.heatmap(
        corr_df.corr(),
        annot=True, fmt=".2f", cmap="RdBu_r", center=0,
        square=True, linewidths=0.5, ax=ax,
    )
    ax.set_title("Pearson correlation – pooled CPET metrics")
    fig.tight_layout()
    fn._save_figure(fig, output_base, "corr_heatmap.png")
    plt.show()
    plt.close(fig)
else:
    print("No telemetry available for correlation analysis.")


### 6e. V-slope analysis (VCO2 vs VO2)

The V-slope method plots VCO₂ against VO₂ during incremental exercise.
Below the first ventilatory threshold (VT1 / AT) the relationship is roughly linear
with slope ≈ 1 (RER < 1).  Above VT1 excess CO₂ from bicarbonate buffering causes
VCO₂ to rise more steeply.  A 45° reference line (slope = 1) is shown in grey.


In [ ]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="VE/VCO2 plots"):
    vo2_col = fn._find_column(telemetry_df, ["VO2", "V'O2", "vo2"])
    vco2_col = fn._find_column(telemetry_df, ["VCO2", "V'CO2", "vco2"])
    stage_col = fn._find_column(telemetry_df, ["Stage"])
    if vo2_col is None or vco2_col is None:
        print(f"{patient_id}: VO2 or VCO2 column not found, skipping V-slope.")
        continue

    tdf = telemetry_df[[vo2_col, vco2_col]]
    if stage_col:
        tdf = telemetry_df[[vo2_col, vco2_col, stage_col]]
    tdf = tdf.apply(lambda s: pd.to_numeric(s, errors="coerce") if s.name != stage_col else s)
    tdf = tdf.dropna(subset=[vo2_col, vco2_col])

    fig, ax = plt.subplots(figsize=(6, 5))
    if stage_col and stage_col in tdf.columns:
        import seaborn as sns
        palette = dict(zip(
            ["Opwarmen", "Test", "VT1", "VT2", "Herstel"],
            sns.color_palette("tab10", 5),
        ))
        for stage, grp in tdf.groupby(stage_col):
            ax.scatter(grp[vo2_col], grp[vco2_col],
                       label=stage, color=palette.get(stage, "grey"), s=15, alpha=0.7)
    else:
        ax.scatter(tdf[vo2_col], tdf[vco2_col], s=15, alpha=0.7)

    # 45° reference line
    lim_min = min(tdf[vo2_col].min(), tdf[vco2_col].min())
    lim_max = max(tdf[vo2_col].max(), tdf[vco2_col].max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max],
            color="grey", linestyle="--", linewidth=1, label="slope = 1")

    ax.set_xlabel(f"{vo2_col} (mL/min)")
    ax.set_ylabel(f"{vco2_col} (mL/min)")
    ax.set_title(f"{patient_id} – V-slope (VCO₂ vs VO₂)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    patient_out = output_base / patient_id
    patient_out.mkdir(parents=True, exist_ok=True)
    fn._save_figure(fig, patient_out, "vslope.png")
    plt.show()
    plt.close(fig)


### 6f. RER trajectory with VT2 marker

The Respiratory Exchange Ratio (RER = VCO₂/VO₂) rises progressively with exercise
intensity.  RER ≥ 1.0 is the conventional marker for VT2 (respiratory compensation
threshold).  A 5-point rolling mean smooths breath-by-breath noise.


In [ ]:
for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="RER plots"):
    rer_col = fn._find_column(telemetry_df, ["RER", "rer"])
    if rer_col is None:
        print(f"{patient_id}: RER column not found, skipping.")
        continue

    rer = pd.to_numeric(telemetry_df[rer_col], errors="coerce").dropna()
    if rer.empty:
        continue
    rer_smooth = rer.rolling(5, min_periods=1, center=True).mean()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(rer_smooth.values, label="RER (5-pt rolling mean)", linewidth=1.5)
    ax.axhline(1.0, color="red", linestyle="--", linewidth=1, label="RER = 1.0 (VT2)")

    # Annotate first crossing of RER = 1.0
    crossings = rer_smooth[rer_smooth >= 1.0]
    if not crossings.empty:
        first_idx = crossings.index[0]
        pos = rer_smooth.index.get_loc(first_idx)
        ax.axvline(pos, color="orange", linestyle=":", linewidth=1)
        ax.annotate(
            f"VT2 ≈ sample {pos}",
            xy=(pos, 1.0), xytext=(pos + 2, rer_smooth.max() * 0.95),
            arrowprops=dict(arrowstyle="->", color="orange"),
            fontsize=8, color="orange",
        )

    ax.set_xlabel("Sample index")
    ax.set_ylabel("RER")
    ax.set_title(f"{patient_id} – RER trajectory")
    ax.legend(fontsize=8)
    fig.tight_layout()
    patient_out = output_base / patient_id
    patient_out.mkdir(parents=True, exist_ok=True)
    fn._save_figure(fig, patient_out, "rer_trajectory.png")
    plt.show()
    plt.close(fig)


### 6g. Heart-rate recovery (HRR)

Heart-rate recovery is the drop in HR from peak exercise to 1 min (HRR-1) and
2 min (HRR-2) into the recovery (Herstel) stage.  Impaired HRR (< 12 bpm at 1 min)
is associated with increased cardiovascular risk.


In [ ]:
hrr_records = []

for patient_id, telemetry_df in tqdm(processed_telemetry.items(), desc="HRR analysis"):
    hr_col = fn._find_column(telemetry_df, ["HR", "hr", "Heart Rate"])
    stage_col = fn._find_column(telemetry_df, ["Stage"])
    if hr_col is None or stage_col is None:
        continue

    hr = pd.to_numeric(telemetry_df[hr_col], errors="coerce")
    stages = telemetry_df[stage_col]

    # Peak HR during Test / VT1 / VT2 stages
    exercise_mask = stages.isin(["Test", "VT1", "VT2"])
    if exercise_mask.sum() == 0:
        exercise_mask = ~stages.isin(["Opwarmen", "Herstel"])
    peak_hr = hr[exercise_mask].max()

    # Herstel rows in order
    herstel_hr = hr[stages == "Herstel"].dropna().reset_index(drop=True)
    if herstel_hr.empty or np.isnan(peak_hr):
        continue

    # Approximate 1 min and 2 min: data is ~5-second epochs → 12 and 24 samples
    SAMPLES_PER_MIN = 12
    hr_1min = herstel_hr.iloc[min(SAMPLES_PER_MIN - 1, len(herstel_hr) - 1)]
    hr_2min = herstel_hr.iloc[min(2 * SAMPLES_PER_MIN - 1, len(herstel_hr) - 1)]
    hrr_1 = peak_hr - hr_1min
    hrr_2 = peak_hr - hr_2min
    hrr_records.append({
        "patient_id": patient_id,
        "peak_HR": round(peak_hr, 1),
        "HRR-1 (bpm)": round(hrr_1, 1),
        "HRR-2 (bpm)": round(hrr_2, 1),
    })

if hrr_records:
    hrr_df = pd.DataFrame(hrr_records).set_index("patient_id")
    display(hrr_df)

    fig, ax = plt.subplots(figsize=(max(4, len(hrr_records) * 2), 4))
    x = np.arange(len(hrr_df))
    w = 0.35
    ax.bar(x - w / 2, hrr_df["HRR-1 (bpm)"], w, label="HRR-1 (1 min)")
    ax.bar(x + w / 2, hrr_df["HRR-2 (bpm)"], w, label="HRR-2 (2 min)")
    ax.axhline(12, color="red", linestyle="--", linewidth=1, label="HRR-1 threshold (12 bpm)")
    ax.set_xticks(x)
    ax.set_xticklabels(hrr_df.index, rotation=30, ha="right")
    ax.set_ylabel("ΔHR (bpm)")
    ax.set_title("Heart-rate recovery (HRR-1 and HRR-2)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fn._save_figure(fig, output_base, "hrr_bar.png")
    plt.show()
    plt.close(fig)
else:
    print("Insufficient data for HRR calculation (need Herstel stage and HR column).")


## 7. Unsupervised Machine Learning

**PCA** is applied to reduce the high-dimensional telemetry feature space to
a small number of interpretable axes.  The Scree plot guides the choice of
how many components capture the majority of variance (target: >= 90 %).

**K-Means** is then applied to the original scaled space.  K-Means is chosen
because the expected physiological response groups (e.g., low/moderate/high
exertion) are approximately convex and of similar size - well-suited to
centroid-based clustering.  The Elbow plot is used to select K.

**DBSCAN** is applied as a complementary density-based algorithm, which can
identify irregularly shaped clusters and flag outlier observations as noise
(label = -1).  This helps reveal atypical physiological responses that
K-Means would force into a cluster.

Results are visualised in the PCA reduced space and as VO2 vs HR scatter
plots coloured by cluster assignment.


### 7b. Feature preparation


In [ ]:
# Concatenate all patient telemetry for a global ML analysis.
if processed_telemetry:
    all_tele = pd.concat(
        [df.assign(patient_id=pid) for pid, df in processed_telemetry.items()],
        ignore_index=True,
    )

    X_scaled, feature_df, scaler = fn.prepare_features(
        all_tele, drop_cols=["patient_id"]
    )
    stage_labels = all_tele["Stage"].fillna("Unknown").reset_index(drop=True)

    print(f"Feature matrix shape: {X_scaled.shape}")
else:
    print("No telemetry data available. Skipping ML section.")
    X_scaled = None


### 7c. PCA


In [ ]:
if X_scaled is not None and X_scaled.shape[0] > 1:
    pca_model, X_pca = fn.run_pca(X_scaled, n_components=min(10, X_scaled.shape[1]))

    # Scree plot
    fig_scree = fn.plot_scree(pca_model, output_dir=output_base)
    plt.show()
    plt.close(fig_scree)

    # 2-D PCA scatter coloured by trial stage
    fig_pca_2d = fn.plot_pca_scatter(
        X_pca, stage_labels, label_name="Stage", output_dir=output_base
    )
    plt.show()
    plt.close(fig_pca_2d)

    # 3-D PCA scatter coloured by trial stage (when >= 3 PCs available)
    if X_pca.shape[1] >= 3:
        fig_pca_3d = fn.plot_pca_scatter(
            X_pca,
            stage_labels,
            label_name="Stage",
            output_dir=output_base,
            three_d=True,
        )
        plt.show()
        plt.close(fig_pca_3d)
else:
    print("Insufficient data for PCA.")


### 7d. Elbow plot for K-Means


In [ ]:
if X_scaled is not None and X_scaled.shape[0] > 10:
    max_k = min(10, X_scaled.shape[0] - 1)
    fig_elbow = fn.elbow_plot(
        X_scaled, k_range=range(2, max_k + 1), output_dir=output_base
    )
    plt.show()
    plt.close(fig_elbow)
else:
    print("Insufficient data for elbow analysis.")


### 7e. K-Means clustering

Based on the elbow plot above, select `N_CLUSTERS`.


In [ ]:
N_CLUSTERS = 4  # adjust based on the elbow plot

if X_scaled is not None and X_scaled.shape[0] >= N_CLUSTERS:
    km_model, km_labels = fn.run_kmeans(X_scaled, n_clusters=N_CLUSTERS)

    # Determine the best available metric pair for visualisation.
    vo2_col = fn._find_column(feature_df, ["VO2", "vo2", "VO2_max"])
    hr_col = fn._find_column(feature_df, ["HR", "hr", "Heart Rate"])
    x_vis = vo2_col or feature_df.columns[0]
    y_vis = hr_col or feature_df.columns[min(1, len(feature_df.columns) - 1)]

    fig_km = fn.plot_cluster_scatter(
        feature_df,
        km_labels,
        x_col=x_vis,
        y_col=y_vis,
        algorithm_name="K-Means",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_km)

    # PCA scatter coloured by K-Means label
    km_series = pd.Series(km_labels.astype(str), name="KMeans_Cluster")
    fig_pca_km = fn.plot_pca_scatter(
        X_pca,
        km_series,
        label_name="K-Means Cluster",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_pca_km)
else:
    print("Insufficient data for K-Means.")


**Cluster interpretation:** Each cluster corresponds to a distinct
physiological response regime identifiable in the VO2 vs HR space.
Clusters with high VO2 and high HR typically represent maximal or near-maximal
exertion (Test / VT2 stages), while low-VO2 / low-HR clusters correspond to
warm-up (Opwarmen) and recovery (Herstel).


### 7f. DBSCAN clustering

DBSCAN does not require specifying K in advance and naturally identifies noise
points (label = -1), making it suitable for detecting physiologically
atypical observations.


In [ ]:
if X_scaled is not None and X_scaled.shape[0] >= 10:
    db_model, db_labels = fn.run_dbscan(X_scaled, eps=1.0, min_samples=5)
    n_clusters_found = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = int(np.sum(db_labels == -1))
    print(f"DBSCAN found {n_clusters_found} cluster(s) and {n_noise} noise point(s).")

    fig_db = fn.plot_cluster_scatter(
        feature_df,
        db_labels,
        x_col=x_vis,
        y_col=y_vis,
        algorithm_name="DBSCAN",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_db)
else:
    print("Insufficient data for DBSCAN.")


**DBSCAN interpretation:** Points labelled -1 (noise) are physiological
observations that do not fit any dense neighbourhood and may indicate artefacts
or genuinely extreme responses.  The remaining clusters can be compared to
the K-Means assignment to assess stability of the groupings.


## 8. Output Summary

The `output/` directory now contains:

```
output/
  master_summary.csv
  batch_vo2_by_gender.png
  batch_bmi_vs_vo2.png
  scree.png
  pca_2d_stage.png
  elbow.png
  kmeans_clusters.png
  dbscan_clusters.png
  <patient_id>/
    <patient_id>_telemetry.csv
    <patient_id>_metrics_by_stage.png
```


In [ ]:
# List all files written to the output directory.
for path in sorted(output_base.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_base))


## 9. Notebook Export

Export this notebook to HTML or PDF using `nbconvert`:

**HTML (recommended - no additional LaTeX required):**

```bash
jupyter nbconvert --to html main.ipynb --output output/main_report.html
```

**PDF (requires LaTeX / pandoc to be installed):**

```bash
jupyter nbconvert --to pdf main.ipynb --output output/main_report.pdf
```

Or run programmatically from within the notebook:


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "html",
        "main.ipynb",
        "--output", str(output_base / "main_report.html"),
    ],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("Notebook exported to:", output_base / "main_report.html")
else:
    print("Export failed:", result.stderr)


---

**End of pipeline.**

All outputs adhere to FAIR principles:

- **Findable:** Structured `output/` directory with consistent naming.
- **Accessible:** Standard CSV and PNG formats; HTML report for easy sharing.
- **Interoperable:** Pandas CSV format readable by any tabular data tool.
- **Reusable:** Modular `functions.py` with full docstrings and type hints;
  `requirements.txt` with pinned dependency versions.
